In [26]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold

df = pd.read_csv('course_lead_scoring_2026.csv')

In [27]:
df.isna().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [28]:
cat_feats = df.columns[df.dtypes == 'string']
num_feats = df.columns[df.dtypes != 'string']

In [29]:
df[cat_feats] = df[cat_feats].fillna('NA')
df[num_feats] = df[num_feats].fillna(0.0)
df.isna().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [30]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [31]:
def train(df_train, y_train, C=1.0):
    dicts = df_train[cat_feats + num_feats].to_dict(orient='records')
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(dicts)

    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000)
    model.fit(X_train, y_train)

    return dv, model

def predict(df, dv, model):
    dicts = df[cat_feats + num_feats].to_dict(orient='records')
    X = dv.transform(dicts)
    y_pred = model.predict_proba(X)[:, 1]
    return y_pred

feats_to_check = ['lead_score', 'number_of_courses_viewed', 'interaction_count', 'annual_income']

for feat in feats_to_check:
    auc_score = roc_auc_score(y_train, df_train[feat])
    if auc_score < 0.5:
        auc_score = roc_auc_score(y_train, -df_train[feat])
    print(feat + ": " + str(auc_score))

lead_score: 0.7882254810906102
number_of_courses_viewed: 0.7229894623989618
interaction_count: 0.7649203047563227
annual_income: 0.6081977776250831


In [32]:
cat_feats = list(cat_feats)
num_feats = list(num_feats)
num_feats.remove('converted')

dv, model = train(df_train, y_train, 1.0)

y_pred = predict(df_val, dv, model)


auc = roc_auc_score(y_val, y_pred)
print(round(auc, 3))

0.732


In [33]:
thresholds = np.linspace(0, 1, 101)
scores = []

for t in thresholds:
    predict_pos = (y_pred >= t)
    
    tp = ((predict_pos == 1) & (y_val == 1)).sum()
    fp = ((predict_pos == 1) & (y_val == 0)).sum()
    fn = ((predict_pos == 0) & (y_val == 1)).sum()
    
    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    scores.append((t, p, r))

df_scores = pd.DataFrame(scores, columns=['threshold', 'precision', 'recall'])
df_scores['diff'] = abs(df_scores['precision'] - df_scores['recall'])
df_scores['f1'] = 2 * (df_scores['precision'] * df_scores['recall']) / (df_scores['precision'] + df_scores['recall'])

In [34]:
df_valid = df_scores[(df_scores['precision'] > 0) | (df_scores['recall'] > 0)]

print(df_valid.loc[df_valid['diff'].idxmin()]['threshold'])

0.63


In [35]:
print(df_scores.loc[df_scores['f1'].idxmax()]['threshold'])

0.41000000000000003


In [36]:
for C in [0.000001, 0.001, 1.0]:
    kfold = KFold(n_splits=5, shuffle=True, random_state=1)
    auc_scores = []
    
    for train_idx, val_idx in kfold.split(df_full_train):
        df_train_cv = df_full_train.iloc[train_idx]
        df_val_cv = df_full_train.iloc[val_idx]

        y_train_cv = df_train_cv.converted.values
        y_val_cv = df_val_cv.converted.values

        dv, model = train(df_train_cv, y_train_cv, C=C)
        y_pred_cv = predict(df_val_cv, dv, model)

        auc = roc_auc_score(y_val_cv, y_pred_cv)
        auc_scores.append(auc)

    mean_auc = round(np.mean(auc_scores), 3)
    std_auc = round(np.std(auc_scores), 3)
    
    print(f"C={C:<8} | Mean: {mean_auc} | Std: {std_auc}")

C=1e-06    | Mean: 0.617 | Std: 0.019
C=0.001    | Mean: 0.749 | Std: 0.01
C=1.0      | Mean: 0.732 | Std: 0.007
